In [ ]:
# Cell 1: Import necessary libraries
import numpy as np
import os
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser, RigidTransform, RotationMatrix,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, InverseKinematics,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization, 
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions, 
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder, 
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph, 
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from scipy.spatial import ConvexHull
import mcubes
from functools import partial
import matplotlib.pyplot as plt
from ciris_plant_visualizer import CIrisPlantVisualizer
from ipywidgets import widgets

# 1. Set up the Scene

In [ ]:
# Replace DiagramBuilder with RobotDiagramBuilder
builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

# Add the robot
gripper = parser.AddModels(file_name="my_sdfs/wsg_3dof.sdf")[0]
cap = parser.AddModels(file_name="my_sdfs/bottle_cap.sdf")[0]
obstacle1 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
# obstacle2 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
obstacle3 = parser.AddModels("my_sdfs/obstacle.sdf")[0]

# Set welds
plant.WeldFrames(
    plant.world_frame(), 
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0, 0, 0]))

# Weld the obstacle to the world frame (adjust pose as needed)
obstacle_pose1 = RigidTransform(RotationMatrix(), [0.01, 0.035, 0.02])  # Adjust position
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle1),
    obstacle_pose1)

# obstacle_pose2 = RigidTransform(RotationMatrix(), [-0.01, 0.035, 0.02])  # Adjust position
# plant.WeldFrames(
#     plant.world_frame(),
#     plant.GetFrameByName("obstacle_link", obstacle2),
#     obstacle_pose2)

obstacle_pose3 = RigidTransform(RotationMatrix(), [-0.035, -0.005, 0.02])  # Adjust position
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle3),
    obstacle_pose3)

p_GgraspO = [0, 0, .065]
R_GgraspO = RotationMatrix.MakeXRotation(-np.pi / 2)
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_wsg", gripper),
    RigidTransform(R_GgraspO, p_GgraspO))

# Fix right finger to left finger
right_finger_joint = plant.GetJointByName("right_finger_sliding_joint", gripper)
left_finger_joint = plant.GetJointByName("left_finger_sliding_joint", gripper)

# Set default joint translation to 0.025
# right_finger_joint.set_default_translation(0.025)
left_finger_joint.set_default_translation(-0.025)

plant.Finalize()

print("Number of positions: ", plant.num_positions())

# Cell 3: Initialize the CIrisPlantVisualizer
q_star = np.zeros(plant.num_positions())

# The object we will use to perform our certification
cspace_free_polytope = CspaceFreePolytope(
    plant, 
    scene_graph,
    SeparatingPlaneOrder.kAffine,
    q_star)

visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    cspace_free_polytope,
    viz_role=Role.kIllustration,
    allow_plus_3dof=True
)


visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)

### 1.1. Use Sliders to Visualize the Scene in Meshcat

In [ ]:
sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
# q = np.zeros(plant.num_positions())
q = [-3.14, -0.78, -0.03, 0.03]
def handle_slider_change(change, idx):
    q[idx] = change['new']
    # print(visualizer.check_collision_q_by_ik(q))
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

## 1.2. Setup Grasping and Placement Space

In [ ]:
# Define the 8 corner points of the convex hull
x_bounds = [-3.14, 3.14]
y_bounds = [-1., 1.]
z_bounds = [-0.055, -0.024]

z2_bounds = [0.024, 0.055]

# lower_joint_limits = np.array([x_bounds[0], y_bounds[0], z_bounds[0]])
# upper_joint_limits = np.array([x_bounds[1], y_bounds[1], z_bounds[1]])

lower_joint_limits = np.array([x_bounds[0], y_bounds[0], z_bounds[0], z2_bounds[0]])
upper_joint_limits = np.array([x_bounds[1], y_bounds[1], z_bounds[1], z2_bounds[1]])

z_bounds_grasp = [-0.025, -0.024]

z2_bounds_grasp = [0.024, 0.025]

# Generate all corner points
# placement_points = np.array([[x, y, z] for x in x_bounds for y in y_bounds for z in z_bounds])
# grasp_points = np.array([[x, y, z] for x in x_bounds for y in y_bounds for z in z_bounds_grasp])
###### For 2 dimensional example:
# placement_points = np.array([[x, y] for x in x_bounds for y in z_bounds])
# grasp_points = np.array([[x, y] for x in x_bounds for y in z_bounds_grasp])

placement_points = np.array([[x, y, z, z2] for x in x_bounds for y in y_bounds for z in z_bounds for z2 in z2_bounds])
grasp_points = np.array([[x, y, z, z2] for x in x_bounds for y in y_bounds for z in z_bounds_grasp for z2 in z2_bounds_grasp])

# Compute the convex hull
placement_hull = ConvexHull(placement_points)
grasp_hull = ConvexHull(grasp_points)

# Convert ConvexHull to HPolyhedron
def convex_hull_to_hpolyhedron(hull):
    A = hull.equations[:, :-1]
    b = -hull.equations[:, -1]
    return HPolyhedron(A, b)

placement_polytope = convex_hull_to_hpolyhedron(placement_hull)
grasp_polytope = convex_hull_to_hpolyhedron(grasp_hull)

# 2. Manipulation Planner Class

In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start: Path, markers=(".git", "README.md")) -> Path:
    for candidate in (start, *start.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(f"Could not locate repo root above {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (REPO_ROOT, REPO_ROOT / "algorithms", REPO_ROOT / "algorithms" / "manipulation_planner"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from manipulation_planner import ManipulationPlanner

# 3. Example Usage

In [ ]:
planner = ManipulationPlanner(visualizer, placement_polytope, grasp_polytope, gripper, cap, 50)

In [ ]:
x_init = np.array([-3.14, -0.78, -0.025, 0.055]) # cap, gripper orientation, left finger, right finger
x_goal = np.array([3.14, -0.78, -0.03, 0.04])
path, traj = planner.compute_trajectory(x_init, x_goal)

In [ ]:
planner.plant.SetPositions(planner.plant_context, x_init)
planner.diagram.ForcedPublish(planner.diagram_context)

In [ ]:
traj_path = planner.display_trajectory(traj, plotly=False)